# 🎬 AI Avatar Studio - HeyGen Open-Source no Google Colab
Gere vídeos de avatares falantes ultra-realistas com **sincronia labial e restauração facial de alta fidelidade** gratuitamente usando GPU NVIDIA T4 do Google Colab.

### 📌 Como usar em 3 passos simples:
1. No menu superior, vá em **Ambiente de execução > Alterar tipo de ambiente de execução** e certifique-se de que a **GPU T4** está selecionada.
2. Execute a **Etapa 1** (Instalação e Download dos Modelos).
3. Execute a **Etapa 2** (Iniciar WebUI) e clique no link público `https://xxxx.gradio.live` gerado para abrir a interface no seu navegador.

## ⚙️ Etapa 1: Instalação e Download dos Modelos

In [ ]:
# 1. Verificar GPU NVIDIA com CUDA
!nvidia-smi

# 2. Instalar pacotes de sistema e bibliotecas Python
!apt-get update -y && apt-get install -y ffmpeg
!pip install --upgrade pip
!pip install gradio edge-tts pydub opencv-python imageio imageio-ffmpeg pillow scipy librosa tqdm facexlib basicsr insightface gdown huggingface_hub

# 3. Clonar repositórios essenciais (Wav2Lip & CodeFormer)
import os
if not os.path.exists('Wav2Lip'):
    !git clone https://github.com/Rudrabha/Wav2Lip.git

if not os.path.exists('CodeFormer'):
    !git clone https://github.com/sczhou/CodeFormer.git
    %cd CodeFormer
    !pip install -r requirements.txt
    !python basicsr/setup.py develop
    %cd ..

# 4. Baixar Pesos dos Modelos Pré-treinados de IA
!mkdir -p checkpoints
!mkdir -p Wav2Lip/checkpoints

# Download dos pesos do Wav2Lip-GAN
if not os.path.exists('checkpoints/wav2lip_gan.pth'):
    !wget -O checkpoints/wav2lip_gan.pth https://huggingface.co/Akumzy/Wav2Lip-GAN/resolve/main/wav2lip_gan.pth
    !cp checkpoints/wav2lip_gan.pth Wav2Lip/checkpoints/

# Download do detector facial s3fd
if not os.path.exists('Wav2Lip/face_detection/detection/sfd/s3fd.pth'):
    !wget -O Wav2Lip/face_detection/detection/sfd/s3fd.pth https://huggingface.co/Akumzy/Wav2Lip-GAN/resolve/main/s3fd-619a316848.pth

# Download dos modelos faciais do CodeFormer
%cd CodeFormer
!python scripts/download_pretrained_models.py facelib
!python scripts/download_pretrained_models.py CodeFormer
%cd ..

print('\n✅ Ambiente configurado com sucesso! Prossiga para a Etapa 2.')

## 🚀 Etapa 2: Iniciar a Interface Web (Gradio WebUI)

In [ ]:
import os, sys, asyncio, edge_tts, subprocess, torch
import gradio as gr

AVAILABLE_VOICES = {
    'Português (Brasil) - Antônio (Masculino Natural)': 'pt-BR-AntonioNeural',
    'Português (Brasil) - Francisca (Feminino Natural)': 'pt-BR-FranciscaNeural',
    'Português (Brasil) - Thalita (Feminino Jovem)': 'pt-BR-ThalitaNeural',
    'Inglês (EUA) - Guy (Masculino)': 'en-US-GuyNeural',
    'Inglês (EUA) - Jenny (Feminino)': 'en-US-JennyNeural',
    'Espanhol - Alvaro (Masculino)': 'es-ES-AlvaroNeural',
    'Espanhol - Elvira (Feminino)': 'es-ES-ElviraNeural',
}

async def _synthesize_edge_tts(text, voice, output_path):
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(output_path)
    return output_path

def generate_speech(text, voice_name, output_path):
    os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)
    asyncio.run(_synthesize_edge_tts(text, voice_name, output_path))
    return output_path

def run_wav2lip(face_path, audio_path, output_path='workspace/temp/raw_lip.mp4'):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    cmd = [
        'python', 'Wav2Lip/inference.py',
        '--checkpoint_path', 'checkpoints/wav2lip_gan.pth',
        '--face', face_path,
        '--audio', audio_path,
        '--outfile', output_path,
        '--pads', '0', '10', '0', '0',
        '--resize_factor', '1'
    ]
    subprocess.run(cmd, check=True)
    return output_path

def enhance_video(video_path, fidelity=0.6):
    cmd = [
        'python', 'CodeFormer/inference_codeformer.py',
        '-w', str(fidelity),
        '--input_path', video_path,
        '--bg_upsampler', 'realesrgan',
        '--face_upsample',
        '-o', 'workspace/output'
    ]
    subprocess.run(cmd, check=True)
    enhanced_path = os.path.join('workspace/output/results', os.path.basename(video_path))
    return enhanced_path if os.path.exists(enhanced_path) else video_path

def process_pipeline(image_input, mode, text, voice, audio_input, use_enhance, fidelity):
    if image_input is None:
        raise gr.Error('Por favor, envie uma foto ou vídeo para o avatar!')
    
    os.makedirs('workspace/temp', exist_ok=True)
    os.makedirs('workspace/output', exist_ok=True)
    
    # 1. Gerar ou receber áudio
    if mode == 'Digitar Texto (TTS)':
        if not text.strip():
            raise gr.Error('Digite o texto a ser falado.')
        audio_path = 'workspace/temp/audio.wav'
        voice_id = AVAILABLE_VOICES.get(voice, 'pt-BR-AntonioNeural')
        generate_speech(text, voice_id, audio_path)
    else:
        if not audio_input:
            raise gr.Error('Envie um arquivo de áudio.')
        audio_path = audio_input
        
    # 2. Sincronia Labial
    lip_output = run_wav2lip(image_input, audio_path)
    
    # 3. Restauração Facial em Alta Resolução
    if use_enhance:
        final_video = enhance_video(lip_output, fidelity)
        return final_video
    return lip_output

with gr.Blocks(title='AI Avatar Studio', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🎬 AI Avatar Studio (HeyGen Open-Source)\n### Gere avatares realistas com sincronia labial e dentes/pele restaurados em alta definição.')
    
    with gr.Row():
        with gr.Column():
            gr.Markdown('### 1. Imagem / Vídeo do Avatar')
            img_in = gr.Image(type='filepath', label='Foto ou Vídeo Base')
            
            gr.Markdown('### 2. Fala / Voz')
            mode_in = gr.Radio(['Digitar Texto (TTS)', 'Enviar Arquivo de Áudio'], value='Digitar Texto (TTS)', label='Modo')
            text_in = gr.Textbox(label='Texto para o Avatar Falar', placeholder='Digite aqui o que o avatar deve falar...', lines=3)
            voice_in = gr.Dropdown(list(AVAILABLE_VOICES.keys()), value='Português (Brasil) - Antônio (Masculino Natural)', label='Voz em Português / Idiomas')
            audio_in = gr.Audio(type='filepath', label='Upload de Áudio Opcional')
            
            gr.Markdown('### 3. Restauração e Nitidez Facial')
            enhance_in = gr.Checkbox(value=True, label='Ativar CodeFormer (Dentes e Olhos Nítidos)')
            fid_in = gr.Slider(0.1, 1.0, value=0.6, step=0.1, label='Fidelidade da Restauração (0.6 recomendado)')
            
            btn = gr.Button('🚀 Gerar Vídeo do Avatar', variant='primary', size='lg')
            
        with gr.Column():
            gr.Markdown('### 4. Resultado Final')
            vid_out = gr.Video(label='Vídeo Renderizado', autoplay=True)
            
    btn.click(process_pipeline, inputs=[img_in, mode_in, text_in, voice_in, audio_in, enhance_in, fid_in], outputs=vid_out)

demo.launch(share=True, debug=True)
